In [0]:
# =====================================================================
# proceso / 04_transform.py
# =====================================================================

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "retail_medallion")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
calendar_clean = (
    spark.table(f"{catalogo}.bronze.calendar_holidays")
    .dropDuplicates(["Date"])
    .withColumn("holiday_date", F.expr("try_to_date(Date, 'yyyy-MM-dd')"))
    .select(F.col("holiday_date"), F.col("Holiday").alias("holiday_name"))
)

bronze_superstore = spark.table(f"{catalogo}.bronze.superstore_raw")

In [0]:
superstore_clean = (
    bronze_superstore
    .dropDuplicates(["Row_ID"]) 
    .filter(F.col("Sales").isNotNull() & (F.col("Sales") > 0))
    .withColumn("order_date", F.expr("try_to_date(Order_Date, 'M/d/yyyy')")) 
    .withColumn("ship_date", F.expr("try_to_date(Ship_Date, 'M/d/yyyy')"))
    .filter(F.col("order_date").isNotNull()) 
    .withColumn("lead_time_days", F.datediff(F.col("ship_date"), F.col("order_date")))
    .withColumn("profit_margin_pct", F.round(F.col("Profit") / F.col("Sales"), 4))
    .join(calendar_clean, F.col("order_date") == calendar_clean.holiday_date, "left")
    .withColumn("is_holiday", F.col("holiday_date").isNotNull())
    .withColumn("_transform_timestamp", F.current_timestamp())
    .select(
        F.col("Order_ID").alias("order_id"),
        F.col("order_date"),
        F.col("Customer_ID").alias("customer_id"),
        F.col("Segment").alias("customer_segment"),
        F.col("Region").alias("region"),
        F.col("Product_ID").alias("product_id"),
        F.col("Category").alias("category"),
        F.col("Sub_Category").alias("sub_category"),
        F.col("Sales").alias("sales_amount"),
        F.col("Quantity").alias("quantity"),
        F.col("Discount").alias("discount_pct"),
        F.col("Profit").alias("profit_amount"),
        F.col("lead_time_days"),
        F.col("profit_margin_pct"),
        F.col("is_holiday"),
        F.col("holiday_name"),
        F.col("_transform_timestamp"),
    )
)

superstore_clean.write.mode("overwrite").insertInto(f"{catalogo}.silver.superstore_clean")
print(f"Silver OK -> {catalogo}.silver.superstore_clean ({superstore_clean.count()} filas)")

In [0]:
bronze_ecommerce = spark.table(f"{catalogo}.bronze.ecommerce_raw")

ecommerce_clean = (
    bronze_ecommerce
    .dropDuplicates(["Date"])
    .filter(F.col("Price").isNotNull() & (F.col("Price") > 0) & F.col("Units_Sold").isNotNull())
    .withColumn("order_date", F.expr("try_to_date(Date, 'd-M-yyyy')"))
    .filter(F.col("order_date").isNotNull())
    .withColumn("sales_amount", F.round(F.col("Price") * F.col("Units_Sold"), 2))
    .withColumn("discount_pct", F.round(F.col("Discount") / 100.0, 4))
    .withColumn(
        "profit_amount",
        F.round(F.col("sales_amount") * (F.lit(1) - F.col("discount_pct")) - F.col("Marketing_Spend"), 2),
    )
    .withColumn("profit_margin_pct", F.round(F.col("profit_amount") / F.col("sales_amount"), 4))
    .join(calendar_clean, F.col("order_date") == calendar_clean.holiday_date, "left")
    .withColumn("is_holiday", F.col("holiday_date").isNotNull())
    .withColumn("_transform_timestamp", F.current_timestamp())
    .select(
        F.col("order_date"),
        F.col("Product_Category").alias("category"),
        F.col("Customer_Segment").alias("customer_segment"),
        F.col("sales_amount"),
        F.col("Units_Sold").alias("units_sold"),
        F.col("discount_pct"),
        F.col("Marketing_Spend").alias("marketing_spend"),
        F.col("profit_amount"),
        F.col("profit_margin_pct"),
        F.col("is_holiday"),
        F.col("holiday_name"),
        F.col("_transform_timestamp"),
    )
)

ecommerce_clean.write.mode("overwrite").insertInto(f"{catalogo}.silver.ecommerce_daily_clean")
print(f"Silver OK -> {catalogo}.silver.ecommerce_daily_clean ({ecommerce_clean.count()} filas)")